# Part 1: Data Cleaning & Validation

This notebook inspects, diagnoses, and cleans the `ohlc_hw` stock-market dataset.

## Objectives
1. Load and inspect the source data.
2. Detect missing values, missing dates, outliers, and inconsistencies.
3. Record data-quality issues before repairs.
4. Repair defensible errors and quarantine unresolved rows.
5. Validate the cleaned data and save `ohlc_hw_cleaned` to SQLite.

The original `ohlc_hw` table is preserved.

## 1.1 Load the data and understanding column definitions

In [1]:

from pathlib import Path
import sqlite3
import numpy as np
import pandas as pd

# Find the project root even if the notebook starts in a subfolder.
def find_database():
    """Locate the course database relative to the current directory."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidate = base / "course_file_export" / "data.db"
        if candidate.is_file():
            return candidate.resolve()

    raise FileNotFoundError(
        "Could not find course_file_export/data.db. "
        "Open the repository folder in VS Code."
    )

DB_PATH = find_database()
print(f"Database: {DB_PATH}")
print(f"Database exists: {DB_PATH.exists()}")


Database: /Users/vandu/UCB-MFE-python-preprogram-2026-winter/course_file_export/data.db
Database exists: True


In [2]:

# Read the source table without modifying the database.
with sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True) as conn:
    df = pd.read_sql_query(
        "SELECT * FROM ohlc_hw ORDER BY ticker, ts",
        conn
    )

print("Dataset dimensions:", df.shape)
print("\nFirst five rows:")
display(df.head())

print("\nDataFrame information:")
df.info()

print("\nDescriptive statistics:")
display(df.describe(include="all").T)

print("\nMissing values by column:")
display(df.isna().sum().rename("missing_count").to_frame())

print("\nDate range:")
print("Start:", df["ts"].min())
print("End:", df["ts"].max())

print("\nNumber of unique tickers:", df["ticker"].nunique())
print("Tickers:")
print(sorted(df["ticker"].dropna().unique()))


Dataset dimensions: (397711, 8)

First five rows:


,ts,open,high,low,close,adj_close,volume,ticker
0,1981-10-19,0.083147,0.083147,0.083147,0.083147,0.064096,20585620,"<a target=""_blank""><span style=""color: blue"">A..."
1,1981-10-20,0.087612,0.087612,0.087612,0.087612,0.067538,35728032,"<a target=""_blank""><span style=""color: blue"">A..."
2,1982-01-20,0.090402,0.090402,0.090402,0.090402,0.069689,25827224,"<a target=""_blank""><span style=""color: blue"">A..."
3,1982-02-11,0.082589,0.082589,0.082589,0.082589,0.063666,24528024,"<a target=""_blank""><span style=""color: blue"">A..."
4,1982-03-26,0.072545,0.072545,0.072545,0.072545,0.055923,50780848,"<a target=""_blank""><span style=""color: blue"">A..."



DataFrame information:
<class 'pandas.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   ts         397711 non-null  str    
 1   open       397711 non-null  float64
 2   high       397711 non-null  float64
 3   low        397711 non-null  float64
 4   close      397711 non-null  float64
 5   adj_close  377856 non-null  float64
 6   volume     397711 non-null  int64  
 7   ticker     397711 non-null  str    
dtypes: float64(5), int64(1), str(2)
memory usage: 31.6 MB

Descriptive statistics:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
ts,397711,15868,2020-11-24,40,NaN,NaN,NaN,NaN,NaN,NaN,NaN
open,397711.0,NaN,NaN,NaN,46.285479,67.132594,0.0,4.209938,19.85,57.53,604.26
high,397711.0,NaN,NaN,NaN,46.74766,67.77821,0.005208,4.24,20.06,58.13,608.63
low,397711.0,NaN,NaN,NaN,45.816279,66.477178,0.004801,4.16,19.63,56.95,597.88
close,397711.0,NaN,NaN,NaN,46.292903,67.141879,0.005208,4.2,19.86,57.53,605.4
adj_close,377856.0,NaN,NaN,NaN,51.65225,194.584045,0.002945,1.39,11.96,43.77,10442.510783
volume,397711.0,NaN,NaN,NaN,30977733.689946,126623650.175253,0.0,1730700.0,4648200.0,11023796.5,9276606399.0
ticker,397711,150,PG,15450,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Missing values by column:


,missing_count
ts,0
open,0
high,0
low,0
close,0
adj_close,19855
volume,0
ticker,0



Date range:
Start: 1962-01-02
End: 2024-12-06

Number of unique tickers: 150
Tickers:
['<a target="_blank"><span style="color: blue">AAPL</span></a>', '<a target="_blank"><span style="color: blue">AMGN</span></a>', '<a target="_blank"><span style="color: blue">AMZN</span></a>', '<a target="_blank"><span style="color: blue">AXP</span></a>', '<a target="_blank"><span style="color: blue">BA</span></a>', '<a target="_blank"><span style="color: blue">CAT</span></a>', '<a target="_blank"><span style="color: blue">CRM</span></a>', '<a target="_blank"><span style="color: blue">CSCO</span></a>', '<a target="_blank"><span style="color: blue">CVX</span></a>', '<a target="_blank"><span style="color: blue">DIS</span></a>', '<a target="_blank"><span style="color: blue">GS</span></a>', '<a target="_blank"><span style="color: blue">HD</span></a>', '<a target="_blank"><span style="color: blue">HON</span></a>', '<a target="_blank"><span style="color: blue">IBM</span></a>', '<a target="_blank"><span sty

In [3]:
column_definitions = pd.DataFrame({
    "column": [
        "ticker", "ts", "open", "high",
        "low", "close", "adj_close", "volume"
    ],
    "meaning": [
        "Stock ticker symbol",
        "Trading date",
        "Opening unadjusted price",
        "Highest unadjusted price during the period",
        "Lowest unadjusted price during the period",
        "Closing unadjusted price",
        "Closing price adjusted for applicable corporate actions",
        "Trading volume"
    ]
})

display(column_definitions)

,column,meaning
0,ticker,Stock ticker symbol
1,ts,Trading date
2,open,Opening unadjusted price
3,high,Highest unadjusted price during the period
4,low,Lowest unadjusted price during the period
5,close,Closing unadjusted price
6,adj_close,Closing price adjusted for applicable corporat...
7,volume,Trading volume


### Initial Data Inspection

The `ohlc_hw` table was successfully loaded from the SQLite database. The dataset contains **397,711 rows and 8 columns**, covering the period from January 2, 1962, to December 6, 2024, across 150 unique tickers.

The dataset contains daily stock market information, including opening, high, low, closing, and adjusted closing prices, trading volume, ticker symbols, and trading dates.

The initial inspection revealed the following:

- **Missing values:** The `adj_close` column contains 19,855 missing values, while the remaining columns have no missing values.
- **Invalid prices:** The minimum values of `open` and `close` are zero, requiring further investigation.
- **Ticker formatting:** The ticker symbols contain HTML markup and need to be normalized before further analysis.
- **Data types:** The `ts` column is currently stored as a string and will be converted to a datetime format for date-based operations.
- **Reference data:** The database also contains an `ohlc` table, which will be used to investigate missing trading dates and reconstruct adjusted closing prices where a reliable adjustment factor is available.

These findings are preliminary. Further validation will check for missing trading dates, extreme returns, impossible price relationships, duplicate ticker-date combinations, and volume anomalies before any cleaning is performed.

The original `ohlc_hw` table will be preserved, and all detected issues will be documented before repairs are applied.

## 1.2 Detect Data Quality Issues

In [4]:

import re
import html

def normalize_ticker(value):
    """Extract a plain ticker symbol from an HTML-formatted ticker."""
    if pd.isna(value):
        return value

    value = html.unescape(str(value))
    value = re.sub(r"<[^>]*>", "", value)
    return value.strip()

# Preserve the initially loaded data.
raw_df = df.copy()

# Normalize the working dataset.
df["ticker"] = df["ticker"].map(normalize_ticker)
df["ts"] = pd.to_datetime(df["ts"], errors="coerce")

price_cols = ["open", "high", "low", "close", "adj_close"]

for col in price_cols + ["volume"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Dataset shape:", df.shape)
print("Date range:", df["ts"].min(), "to", df["ts"].max())
print("Number of tickers:", df["ticker"].nunique())
print("Example tickers:", sorted(df["ticker"].dropna().unique())[:20])
print(
    "Duplicate ticker-date keys:",
    df.duplicated(["ticker", "ts"]).sum()
)


Dataset shape: (397711, 8)
Date range: 1962-01-02 00:00:00 to 2024-12-06 00:00:00
Number of tickers: 30
Example tickers: ['AAPL', 'AMGN', 'AMZN', 'AXP', 'BA', 'CAT', 'CRM', 'CSCO', 'CVX', 'DIS', 'GS', 'HD', 'HON', 'IBM', 'JNJ', 'JPM', 'KO', 'MCD', 'MMM', 'MRK']
Duplicate ticker-date keys: 29460


In [5]:

with sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True) as conn:
    reference_df = pd.read_sql_query(
        "SELECT ts, ticker FROM ohlc",
        conn
    )

reference_df["ts"] = pd.to_datetime(
    reference_df["ts"], errors="coerce"
)
reference_df["ticker"] = reference_df["ticker"].map(normalize_ticker)

homework_tickers = set(df["ticker"].dropna().unique())
reference_tickers = set(reference_df["ticker"].dropna().unique())

print("Reference rows:", len(reference_df))
print("Reference tickers:", len(reference_tickers))
print("Homework tickers:", len(homework_tickers))
print("Tickers absent from reference:", 
      sorted(homework_tickers - reference_tickers))


Reference rows: 368251
Reference tickers: 30
Homework tickers: 30
Tickers absent from reference: []


### Data Preparation

The dataset was successfully prepared by normalizing ticker symbols, converting dates to datetime format, and converting price and volume columns to numeric types.

The dataset contains **397,711 rows across 30 tickers**, all of which are available in the reference `ohlc` table. The duplicate-key check identified **29,460 rows belonging to duplicated `(ticker, ts)` combinations**, which require further investigation before removal.

The reference table will be used to detect missing trading dates. No data has been deleted or repaired at this stage.

## Duplicate Observation Investigation

In [6]:

# Identify every row belonging to a duplicate ticker-date key.
duplicate_mask = df.duplicated(["ticker", "ts"], keep=False)

duplicate_rows = df.loc[duplicate_mask].sort_values(
    ["ticker", "ts"]
).copy()

# Count rows per duplicated key.
duplicate_groups = (
    duplicate_rows.groupby(["ticker", "ts"], dropna=False)
    .size()
    .reset_index(name="row_count")
    .sort_values("row_count", ascending=False)
)

print("Rows in duplicate groups:", len(duplicate_rows))
print("Number of duplicate ticker-date groups:", len(duplicate_groups))
print("Largest duplicate group:", duplicate_groups["row_count"].max())

display(duplicate_groups.head(10))
display(duplicate_rows.head(10))


Rows in duplicate groups: 58920
Number of duplicate ticker-date groups: 29460
Largest duplicate group: 2


,ticker,ts,row_count
0,AAPL,1981-01-02,2
19650,MRK,1972-10-06,2
19648,MRK,1972-09-08,2
19647,MRK,1972-08-22,2
19646,MRK,1972-08-21,2
19645,MRK,1972-08-10,2
19644,MRK,1972-07-20,2
19643,MRK,1972-07-13,2
19642,MRK,1972-06-26,2
19641,MRK,1972-06-09,2


,ts,open,high,low,close,adj_close,volume,ticker
39947,1981-01-02,0.15402,0.15402,0.15402,0.15402,0.118730,21660820,AAPL
39948,1981-01-02,0.15402,0.15402,0.15402,0.15402,0.118730,21660820,AAPL
39949,1981-01-05,0.15067,0.15067,0.15067,0.15067,0.116150,35728032,AAPL
39950,1981-01-05,0.15067,0.15067,0.15067,0.15067,0.116150,35728032,AAPL
39974,1981-02-10,0.12165,0.12165,0.12165,0.12165,0.093779,18345616,AAPL
39975,1981-02-10,0.12165,0.12165,0.12165,0.12165,0.093779,18345616,AAPL
39986,1981-02-27,0.11830,0.11830,0.11830,0.11830,0.091197,14761612,AAPL
39987,1981-02-27,0.11830,0.11830,0.11830,0.11830,0.091197,14761612,AAPL
39991,1981-03-05,0.11551,0.11551,0.11551,0.11551,0.089046,5376004,AAPL
39992,1981-03-05,0.11551,0.11551,0.11551,0.11551,0.089046,5376004,AAPL


In [7]:

value_cols = [
    "open", "high", "low", "close", "adj_close", "volume"
]

# Check for exact duplicate records across all columns.
print("Exact duplicate rows:", df.duplicated().sum())

# Check whether price or volume values differ within duplicate groups.
variation = (
    duplicate_rows.groupby(["ticker", "ts"])[value_cols]
    .nunique(dropna=False)
)

conflicting_groups = variation.gt(1).any(axis=1)

print("Duplicate groups with conflicting values:",
      int(conflicting_groups.sum()))
print("Duplicate groups with identical price/volume values:",
      int((~conflicting_groups).sum()))


Exact duplicate rows: 29460
Duplicate groups with conflicting values: 0
Duplicate groups with identical price/volume values: 29460


### Duplicate Investigation

The dataset contains 29,460 duplicated `(ticker, ts)` groups, each with two identical observations. All 29,460 groups have matching price and volume values, with no conflicting records found.

These duplicates are redundant observations rather than conflicting data. During cleaning, one copy of each exact duplicate will be retained to ensure unique ticker-date keys. The original dataset remains unchanged during this investigation.

## Data Quality Detection Functions

In [8]:
# Each issue is stored as a separate row in this list.
issues = []

def add_issue(ticker, issue_type, issue_date, description, severity):
    """Append a finding to the data-quality issue list."""
    issues.append({
        "ticker": ticker,
        "issue_type": issue_type,
        "issue_date": (
            pd.Timestamp(issue_date).strftime("%Y-%m-%d")
            if pd.notna(issue_date) else None
        ),
        "description": description,
        "severity": severity
    })


In [9]:
def detect_missing_values(data):
    """Record missing values in each required data column."""
    for col in data.columns:
        for idx in data.index[data[col].isna()]:
            row = data.loc[idx]
            severity = (
                "high" if col in ["ticker", "ts", "close"]
                else "medium"
            )
            add_issue(
                row.get("ticker"),
                "missing_value",
                row.get("ts"),
                f"Missing value in column '{col}'",
                severity
            )


def detect_duplicate_keys(data):
    """Flag every row involved in a duplicate ticker-date key."""
    mask = data.duplicated(["ticker", "ts"], keep=False)

    for _, row in data.loc[mask].iterrows():
        add_issue(
            row["ticker"],
            "duplicate_key",
            row["ts"],
            "Duplicate (ticker, ts) key; investigate before removal",
            "high"
        )


In [10]:
def detect_invalid_prices(data):
    """Flag non-finite/nonpositive prices and impossible OHLC bounds."""
    for col in ["open", "high", "low", "close", "adj_close"]:
        values = data[col].to_numpy(dtype=float)
        mask = ~np.isfinite(values) | (values <= 0)

        for idx in data.index[mask]:
            row = data.loc[idx]
            add_issue(
                row["ticker"], "invalid_price", row["ts"],
                f"{col} is non-finite or nonpositive: {row[col]}",
                "high"
            )

    checks = [
        ("high < close", data["high"] < data["close"]),
        ("low > close", data["low"] > data["close"]),
        ("low > open", data["low"] > data["open"]),
        ("high < open", data["high"] < data["open"]),
        ("low > high", data["low"] > data["high"])
    ]

    for description, mask in checks:
        for idx in data.index[mask.fillna(False)]:
            row = data.loc[idx]
            add_issue(
                row["ticker"], "impossible_ohlc", row["ts"],
                f"Impossible price relationship: {description}",
                "high"
            )


In [11]:

def detect_volume_issues(data):
    """Flag invalid volume and record zero-volume observations."""
    values = data["volume"].to_numpy(dtype=float)
    invalid = ~np.isfinite(values) | (values < 0)

    for idx in data.index[invalid]:
        row = data.loc[idx]
        add_issue(
            row["ticker"], "invalid_volume", row["ts"],
            f"Volume is negative or non-finite: {row['volume']}",
            "medium"
        )

    for idx in data.index[values == 0]:
        row = data.loc[idx]
        add_issue(
            row["ticker"], "zero_volume", row["ts"],
            "Zero volume; investigate whether the observation is stale",
            "low"
        )


In [12]:

def detect_extreme_returns(data):
    """Flag simple returns more than 3 sample standard deviations from the mean."""
    ordered = data.sort_values(["ticker", "ts"]).copy()
    ordered["simple_return"] = (
        ordered.groupby("ticker")["close"].pct_change(fill_method=None)
    )

    for ticker, group in ordered.groupby("ticker", sort=False):
        returns = group["simple_return"]
        valid = returns[np.isfinite(returns)]

        if len(valid) < 2:
            continue

        mean_return = valid.mean()
        std_return = valid.std(ddof=1)

        # Do not divide by zero or flag undefined returns.
        if not np.isfinite(std_return) or std_return == 0:
            continue

        mask = (
            group["simple_return"].notna()
            & np.isfinite(group["simple_return"])
            & ((group["simple_return"] - mean_return).abs() > 3 * std_return)
        )

        for _, row in group.loc[mask].iterrows():
            add_issue(
                ticker, "extreme_return", row["ts"],
                f"Simple return {row['simple_return']:.4f}; "
                f"ticker mean={mean_return:.4f}, sample std={std_return:.4f}. "
                "Investigate before classifying as an error.",
                "medium"
            )


In [13]:
# Reset the report before running the real-data detectors.
issues = []

detect_missing_values(df)
detect_duplicate_keys(df)
detect_invalid_prices(df)
detect_volume_issues(df)
detect_extreme_returns(df)

quality_report = pd.DataFrame(
    issues,
    columns=[
        "ticker", "issue_type", "issue_date",
        "description", "severity"
    ]
)

print("Total quality issues recorded:", len(quality_report))
display(
    quality_report.groupby(["issue_type", "severity"])
    .size()
    .rename("count")
    .reset_index()
    .sort_values("count", ascending=False)
)

display(quality_report.head(20))


Total quality issues recorded: 104462


,issue_type,severity,count
0,duplicate_key,high,58920
3,invalid_price,high,19860
4,missing_value,medium,19855
1,extreme_return,medium,5741
5,zero_volume,low,81
2,impossible_ohlc,high,5


,ticker,issue_type,issue_date,description,severity
0,AAPL,missing_value,1984-11-06,Missing value in column 'adj_close',medium
1,AAPL,missing_value,1985-09-12,Missing value in column 'adj_close',medium
2,AAPL,missing_value,2003-06-27,Missing value in column 'adj_close',medium
3,AAPL,missing_value,2007-01-25,Missing value in column 'adj_close',medium
4,AAPL,missing_value,2013-06-05,Missing value in column 'adj_close',medium
5,AAPL,missing_value,2013-08-28,Missing value in column 'adj_close',medium
6,AAPL,missing_value,2017-04-28,Missing value in column 'adj_close',medium
7,AAPL,missing_value,2017-04-28,Missing value in column 'adj_close',medium
8,AAPL,missing_value,2020-01-09,Missing value in column 'adj_close',medium
9,AAPL,missing_value,2020-07-27,Missing value in column 'adj_close',medium


### Initial Data Quality Report

The initial checks recorded **104,462 data-quality issues**, including duplicate keys, invalid prices, missing adjusted closing prices, extreme returns, zero-volume observations, and impossible OHLC relationships.

Duplicate keys account for 58,920 findings, while 19,855 missing-value findings relate to `adj_close`. Extreme returns are flagged for further investigation rather than automatically classified as errors. These results are preliminary because duplicate observations can contribute multiple findings.

No data has been repaired or deleted. The findings will guide the cleaning process, followed by validation to confirm that the retained dataset satisfies the required quality checks.

## Missing Trading Day Detection

In [14]:

def detect_missing_trading_days(data, reference):
    """
    Detect missing reference trading dates for each ticker.

    The comparison is limited to the ticker's observed homework
    history. Each missing date becomes a separate issue.
    """
    missing_records = []

    observed = data.dropna(subset=["ticker", "ts"])

    for ticker, group in observed.groupby("ticker"):
        start = group["ts"].min()
        end = group["ts"].max()

        expected_dates = set(
            reference.loc[
                (reference["ticker"] == ticker)
                & reference["ts"].between(start, end),
                "ts"
            ].dropna().unique()
        )

        actual_dates = set(group["ts"].dropna().unique())

        missing_dates = sorted(expected_dates - actual_dates)

        for missing_date in missing_dates:
            missing_records.append({
                "ticker": ticker,
                "issue_date": pd.Timestamp(missing_date).strftime("%Y-%m-%d")
            })

            add_issue(
                ticker,
                "missing_trading_day",
                missing_date,
                f"Ticker {ticker} is missing a reference trading day "
                f"within its observed history ({start.date()} to {end.date()})",
                "medium"
            )

    return pd.DataFrame(
        missing_records,
        columns=["ticker", "issue_date"]
    )


In [15]:

toy_reference = pd.DataFrame({
    "ticker": ["TEST", "TEST", "TEST"],
    "ts": pd.to_datetime([
        "2024-01-02", "2024-01-03", "2024-01-04"
    ])
})

# The January 3 observation is deliberately missing.
toy_observed = pd.DataFrame({
    "ticker": ["TEST", "TEST"],
    "ts": pd.to_datetime(["2024-01-02", "2024-01-04"])
})

# Isolate the test findings from the real-data report.
saved_issues = issues
issues = []

toy_missing = detect_missing_trading_days(
    toy_observed, toy_reference
)

display(toy_missing)
assert toy_missing["issue_date"].tolist() == ["2024-01-03"]

issues = saved_issues
print("Missing-date test passed.")


,ticker,issue_date
0,TEST,2024-01-03


Missing-date test passed.


In [16]:

missing_trading_days = detect_missing_trading_days(df, reference_df)

print("Missing reference trading dates:", len(missing_trading_days))
print("\nMissing dates by ticker:")
display(
    missing_trading_days.groupby("ticker")
    .size()
    .rename("missing_trading_days")
    .sort_values(ascending=False)
    .reset_index()
)

print("\nExamples:")
display(missing_trading_days.head(20))


Missing reference trading dates: 0

Missing dates by ticker:


,ticker,missing_trading_days



Examples:


,ticker,issue_date


### Missing Trading-Day Detection — Findings

The missing-trading-day detector was tested on a small example with a deliberately omitted reference date, and the test passed.

The full dataset was then compared against the supplied reference table for each ticker, restricted to its observed date range. Dates present in the reference data but absent from the homework dataset were recorded as `missing_trading_day` issues.

Missing dates are reported rather than filled with invented observations. The results will be used to guide the subsequent cleaning and validation steps.

## Consolidated Data Quality Report

In [17]:
quality_report = pd.DataFrame(
    issues,
    columns=[
        "ticker",
        "issue_type",
        "issue_date",
        "description",
        "severity",
    ],
)

print("Total quality findings:", len(quality_report))
print("Unique issue types:", quality_report["issue_type"].nunique())

print("\nFindings by issue type:")
display(
    quality_report.groupby("issue_type")
    .size()
    .rename("count")
    .sort_values(ascending=False)
    .reset_index()
)

print("\nFindings by severity:")
display(
    quality_report.groupby("severity")
    .size()
    .rename("count")
    .reindex(["high", "medium", "low"])
    .fillna(0)
    .astype(int)
    .reset_index()
)

print("\nFindings by ticker:")
display(
    quality_report.groupby("ticker")
    .size()
    .rename("issue_count")
    .sort_values(ascending=False)
    .reset_index()
    .head(15)
)

Total quality findings: 104462
Unique issue types: 6

Findings by issue type:


,issue_type,count
0,duplicate_key,58920
1,invalid_price,19860
2,missing_value,19855
3,extreme_return,5741
4,zero_volume,81
5,impossible_ohlc,5



Findings by severity:


,severity,count
0,high,78785
1,medium,25596
2,low,81



Findings by ticker:


,ticker,issue_count
0,KO,4578
1,HON,4562
2,MRK,4556
3,JNJ,4550
4,IBM,4527
5,CAT,4525
6,DIS,4521
7,PG,4461
8,CVX,4444
9,BA,4412


In [18]:
required_columns = [
    "ticker",
    "issue_type",
    "issue_date",
    "description",
    "severity",
]

assert quality_report.columns.tolist() == required_columns
assert quality_report["issue_type"].notna().all()
assert quality_report["severity"].isin(
    ["high", "medium", "low"]
).all()

print("Quality report validation passed.")

Quality report validation passed.


### Quality Report — Interpretation

The consolidated report provides an overview of data-quality issues by category, severity, and ticker. High-severity findings require particular attention because they may affect price validity or the uniqueness of ticker-date observations. Medium-severity findings require investigation, while low-severity findings are retained for review.

The counts represent findings, not necessarily distinct affected rows. One observation may generate multiple findings, and a duplicate ticker-date group may generate findings for both copies.

The report will be retained as a record of the initial data quality before cleaning. After repairs, the validation checks will be rerun to assess the effectiveness of the cleaning process.

## Unit test the Data-Quality Detectors

In [19]:
toy_data = pd.DataFrame({
    "ticker": ["TEST"] * 4,
    "ts": pd.to_datetime([
        "2024-01-02",
        "2024-01-03",
        "2024-01-04",
        "2024-01-05",
    ]),
    "open":      [100.0, 102.0, 105.0, 108.0],
    "high":      [105.0, 101.0, 110.0, 112.0],
    "low":       [98.0,  99.0, 103.0, 106.0],
    "close":     [103.0, 100.0, 108.0, 110.0],
    "adj_close": [103.0, np.nan, 108.0, 110.0],
    "volume":    [1000, 0, 1500, 1200],
})

# Inject a nonpositive price and an impossible OHLC relationship.
toy_data.loc[2, "open"] = 0.0

saved_issues = issues
issues = []

detect_missing_values(toy_data)
detect_invalid_prices(toy_data)
detect_volume_issues(toy_data)

toy_report = pd.DataFrame(
    issues,
    columns=[
        "ticker",
        "issue_type",
        "issue_date",
        "description",
        "severity",
    ],
)

display(toy_report)

assert (toy_report["issue_type"] == "missing_value").any()
assert (toy_report["issue_type"] == "invalid_price").any()
assert (toy_report["issue_type"] == "zero_volume").any()
assert (toy_report["issue_type"] == "impossible_ohlc").any()

issues = saved_issues

print("Synthetic detector tests passed.")

,ticker,issue_type,issue_date,description,severity
0,TEST,missing_value,2024-01-03,Missing value in column 'adj_close',medium
1,TEST,invalid_price,2024-01-04,open is non-finite or nonpositive: 0.0,high
2,TEST,invalid_price,2024-01-03,adj_close is non-finite or nonpositive: nan,high
3,TEST,impossible_ohlc,2024-01-04,Impossible price relationship: low > open,high
4,TEST,impossible_ohlc,2024-01-03,Impossible price relationship: high < open,high
5,TEST,zero_volume,2024-01-03,Zero volume; investigate whether the observati...,low


Synthetic detector tests passed.


### Synthetic Detector Tests — Findings

The detectors were tested on a synthetic dataset containing deliberately introduced data-quality problems. The tests verify that missing values, nonpositive prices, zero-volume observations, and impossible OHLC relationships are identified as expected.

The synthetic findings were kept separate from the original dataset's quality report. No original observations were changed during testing.

## Duplicate and Extremem-Return Detector Tests

In [20]:
# Test duplicate ticker-date detection.
toy_duplicates = pd.DataFrame({
    "ticker": ["TEST", "TEST", "TEST"],
    "ts": pd.to_datetime([
        "2024-01-02",
        "2024-01-02",
        "2024-01-03",
    ]),
    "open": [100.0, 100.0, 101.0],
    "high": [105.0, 105.0, 106.0],
    "low": [98.0, 98.0, 99.0],
    "close": [103.0, 103.0, 104.0],
    "adj_close": [103.0, 103.0, 104.0],
    "volume": [1000, 1000, 1100],
})

saved_issues = issues
issues = []

detect_duplicate_keys(toy_duplicates)

duplicate_test_report = pd.DataFrame(issues)

assert len(duplicate_test_report) == 2
assert (duplicate_test_report["issue_type"] == "duplicate_key").all()

issues = saved_issues

print("Duplicate detector test passed.")

Duplicate detector test passed.


In [21]:
toy_returns = pd.DataFrame({
    "ticker": ["TEST"] * 20,
    "ts": pd.date_range("2024-01-02", periods=20, freq="D"),
    "close": [
        100, 100.2, 100.1, 100.3, 100.2,
        100.4, 100.3, 100.5, 100.4, 100.6,
        100.5, 100.7, 100.6, 100.8, 100.7,
        100.9, 100.8, 101.0, 100.9, 150.0
    ],
})

saved_issues = issues
issues = []

detect_extreme_returns(toy_returns)

extreme_test_report = pd.DataFrame(
    issues,
    columns=[
        "ticker",
        "issue_type",
        "issue_date",
        "description",
        "severity",
    ],
)

display(extreme_test_report)

if not extreme_test_report.empty:
    assert (
        extreme_test_report["issue_type"] == "extreme_return"
    ).any()
    print("Extreme-return detector test passed.")
else:
    print(
        "No extreme return was flagged. "
        "Review the detector threshold and test data."
    )

issues = saved_issues

,ticker,issue_type,issue_date,description,severity
0,TEST,extreme_return,2024-01-21,"Simple return 0.4866; ticker mean=0.0261, samp...",medium


Extreme-return detector test passed.


### Duplicate and Extreme-Return Tests — Findings

The duplicate detector successfully identified both observations belonging to a duplicated ticker-date key in the synthetic dataset.

The extreme-return detector successfully flagged the deliberately introduced large price movement on January 21, 2024. The observation had a simple return of approximately 48.66%, substantially above the ticker's estimated mean return.

Extreme returns are treated as medium-severity alerts because large price movements can reflect legitimate market events, corporate actions, or data errors. These observations require further investigation before any repair or removal.

The synthetic tests were isolated from the original dataset's quality report, and no original market observations were modified.

## Data Cleaning

### Step 1: Remove Exact duplicate observations

In [22]:
# Preserve the prepared dataset and create a separate cleaning copy.
df_before_cleaning = df.copy(deep=True)
df_clean = df.copy(deep=True)

rows_before = len(df_clean)

# Remove only rows that are identical across every column.
exact_duplicates = df_clean.duplicated(keep="first")
duplicate_rows_removed = int(exact_duplicates.sum())

df_clean = df_clean.drop_duplicates(keep="first").copy()

rows_after = len(df_clean)

print(f"Rows before cleaning: {rows_before:,}")
print(f"Exact duplicate rows removed: {duplicate_rows_removed:,}")
print(f"Rows after duplicate removal: {rows_after:,}")
print(f"Rows retained: {rows_after / rows_before:.2%}")

# Confirm that no exact duplicates remain.
assert not df_clean.duplicated().any()

# Confirm that the original prepared DataFrame was not modified.
assert len(df) == rows_before

print("\nExact-duplicate removal validation passed.")

Rows before cleaning: 397,711
Exact duplicate rows removed: 29,460
Rows after duplicate removal: 368,251
Rows retained: 92.59%

Exact-duplicate removal validation passed.


### Exact Duplicate Removal — Findings

Exact duplicate rows were removed from a separate working copy of the dataset, retaining one copy of each identical observation.

The original prepared dataset remains unchanged. The number of rows removed and retained was recorded to support reproducibility and before-and-after comparisons.

Duplicate ticker-date keys will be checked separately because two observations can share a ticker and date without being identical across all columns.

## Step 2: Validate Ticker-Date Uniqueness

In [23]:
duplicate_key_mask = df_clean.duplicated(
    subset=["ticker", "ts"],
    keep=False,
)

remaining_duplicate_rows = int(duplicate_key_mask.sum())

remaining_duplicate_groups = int(
    df_clean.loc[duplicate_key_mask, ["ticker", "ts"]]
    .drop_duplicates()
    .shape[0]
)

print("Remaining rows in duplicate ticker-date groups:",
      remaining_duplicate_rows)
print("Remaining duplicate ticker-date groups:",
      remaining_duplicate_groups)

if remaining_duplicate_rows:
    display(
        df_clean.loc[duplicate_key_mask]
        .sort_values(["ticker", "ts"])
        .head(20)
    )
else:
    print("Every ticker-date key is unique.")

assert not df_clean.duplicated(
    subset=["ticker", "ts"]
).any()

print("\nTicker-date uniqueness validation passed.")

Remaining rows in duplicate ticker-date groups: 0
Remaining duplicate ticker-date groups: 0
Every ticker-date key is unique.



Ticker-date uniqueness validation passed.


### Ticker-Date Uniqueness — Findings

After exact duplicate removal, the dataset was checked for duplicate `(ticker, ts)` keys. This validation determines whether any multiple observations remain for the same ticker and date.

The dataset must contain at most one observation per ticker-date combination before proceeding to missing-value handling and price validation.

## Step 3: Missing value assessment

In [24]:
price_cols = ["open", "high", "low", "close"]
all_price_cols = price_cols + ["adj_close"]

print("Missing values by column:")
display(
    df_clean.isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

all_raw_prices_missing = df_clean[price_cols].isna().all(axis=1)

print(
    "\nRows with all four raw OHLC prices missing:",
    int(all_raw_prices_missing.sum())
)

print(
    "Rows with at least one missing raw OHLC price:",
    int(df_clean[price_cols].isna().any(axis=1).sum())
)

print(
    "Rows with missing adjusted close:",
    int(df_clean["adj_close"].isna().sum())
)

print(
    "Rows with missing volume:",
    int(df_clean["volume"].isna().sum())
)

Missing values by column:


,missing_count
ts,0
open,0
high,0
low,0
close,0
adj_close,18374
volume,0
ticker,0



Rows with all four raw OHLC prices missing: 0
Rows with at least one missing raw OHLC price: 0
Rows with missing adjusted close: 18374
Rows with missing volume: 0


### Missing-Value Assessment — Findings

Missing values were counted by column after exact duplicate removal. Rows with all four raw OHLC prices missing were identified separately because they may need to be excluded.

Missing raw prices, adjusted closing prices, and volume will be handled using separate documented rules. The reference table will be used to recover adjusted closing prices only when a matching ticker-date observation is available. Unresolved values will be tracked for further review.

## Step 4: Recover Missing Adjusted Closing Prices

In [25]:
# Load adjusted closing prices from the reference table.
with sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True) as conn:
    reference_prices = pd.read_sql_query(
        "SELECT ts, ticker, adj_close FROM ohlc",
        conn,
    )

# Normalize keys and numeric values consistently.
reference_prices["ticker"] = (
    reference_prices["ticker"].map(normalize_ticker)
)
reference_prices["ts"] = pd.to_datetime(
    reference_prices["ts"], errors="coerce"
)
reference_prices["adj_close"] = pd.to_numeric(
    reference_prices["adj_close"], errors="coerce"
)

# Each reference ticker-date must identify at most one row.
assert not reference_prices.duplicated(
    subset=["ticker", "ts"]
).any(), "Reference table contains duplicate ticker-date keys."

# Build a lookup indexed by ticker and date.
reference_adj_lookup = reference_prices.set_index(
    ["ticker", "ts"]
)["adj_close"]

lookup_keys = pd.MultiIndex.from_frame(
    df_clean[["ticker", "ts"]]
)

reference_adj_values = reference_adj_lookup.reindex(
    lookup_keys
).to_numpy()

# Record which values were missing before repair.
missing_before = df_clean["adj_close"].isna()

# Track all adjusted-close repairs and specifically missing-value recoveries.
df_clean["adj_close_repaired"] = False
df_clean["adj_close_missing_repaired"] = False

# Fill only missing values with valid matching reference values.
can_repair = (
    missing_before
    & pd.Series(
        np.isfinite(reference_adj_values)
        & (reference_adj_values > 0),
        index=df_clean.index,
    )
)

df_clean.loc[can_repair, "adj_close"] = (
    reference_adj_values[can_repair.to_numpy()]
)
df_clean.loc[can_repair, "adj_close_repaired"] = True
df_clean.loc[can_repair, "adj_close_missing_repaired"] = True

# Report the outcome.
filled_count = int(can_repair.sum())
unresolved_count = int(df_clean["adj_close"].isna().sum())

print("Missing adj_close before repair:", int(missing_before.sum()))
print("Recovered from reference:", filled_count)
print("Still missing after repair:", unresolved_count)
print(
    "Repair flags recorded:",
    int(df_clean["adj_close_repaired"].sum()),
)

assert int(df_clean["adj_close_repaired"].sum()) == filled_count
assert int(df_clean["adj_close"].isna().sum()) == unresolved_count

Missing adj_close before repair: 18374
Recovered from reference: 18374
Still missing after repair: 0
Repair flags recorded: 18374


### Adjusted Closing Price Recovery — Findings

Missing adjusted closing prices were matched against the reference table using ticker and date. Only valid, positive reference values were used to fill missing entries.

Successful repairs were recorded in the `adj_close_repaired` flag. Values that could not be recovered remain missing and will be tracked for further investigation rather than filled using an unverified adjustment factor.

The original database table remains unchanged.

### Validate and Repair Adjusted-Close Inconsistencies

After recovering missing adjusted-close values, compare the remaining values against the supplied `ohlc` reference table using exact `(ticker, ts)` matches. A mismatch is repaired only when both values are finite and positive and the reference value differs beyond a small numerical tolerance. The original value is retained in `adj_close_original`, and `adj_close_reference_mismatch` marks these repairs. `adj_close_repaired` indicates either a missing-value recovery or a reference-mismatch repair. The original source tables remain unchanged.

In [26]:
# Compare adjusted-close values with the exact ticker-date reference.
# `reference_prices` was loaded and validated in Step 4 above.
reference_adj_lookup = reference_prices.set_index(
    ["ticker", "ts"]
)["adj_close"]

lookup_keys = pd.MultiIndex.from_frame(
    df_clean[["ticker", "ts"]]
)
reference_adj_values = pd.to_numeric(
    reference_adj_lookup.reindex(lookup_keys),
    errors="coerce",
).to_numpy(dtype=float)

current_adj_values = pd.to_numeric(
    df_clean["adj_close"],
    errors="coerce",
).to_numpy(dtype=float)

valid_current = (
    np.isfinite(current_adj_values)
    & (current_adj_values > 0)
)
valid_reference = (
    np.isfinite(reference_adj_values)
    & (reference_adj_values > 0)
)

mismatch_values = (
    valid_current
    & valid_reference
    & ~np.isclose(
        current_adj_values,
        reference_adj_values,
        rtol=1e-6,
        atol=1e-8,
    )
)
reference_mismatch = pd.Series(
    mismatch_values,
    index=df_clean.index,
    name="adj_close_reference_mismatch",
)

# Preserve the pre-repair value only for rows that will be changed.
df_clean["adj_close_original"] = np.nan
df_clean.loc[reference_mismatch, "adj_close_original"] = (
    df_clean.loc[reference_mismatch, "adj_close"]
)
df_clean["adj_close_reference_mismatch"] = reference_mismatch

# Repair only confirmed mismatches with finite, positive reference values.
df_clean.loc[reference_mismatch, "adj_close"] = (
    reference_adj_values[mismatch_values]
)
df_clean.loc[reference_mismatch, "adj_close_repaired"] = True

# Build an audit view with the original and replacement values.
adjusted_close_repair_audit = df_clean.loc[
    reference_mismatch,
    [
        "ticker",
        "ts",
        "adj_close_original",
        "adj_close",
    ],
].rename(columns={"adj_close": "adj_close_reference_replacement"})

print(
    "Adjusted-close reference mismatches repaired:",
    int(reference_mismatch.sum()),
)
if adjusted_close_repair_audit.empty:
    print("No valid adjusted-close mismatches were found.")
else:
    display(adjusted_close_repair_audit.head(25))

# Confirm repaired values match the trusted reference values.
repaired_adj_values = pd.to_numeric(
    df_clean["adj_close"],
    errors="coerce",
).to_numpy(dtype=float)
assert np.isfinite(repaired_adj_values).all()
assert (repaired_adj_values > 0).all()
assert np.allclose(
    repaired_adj_values[mismatch_values],
    reference_adj_values[mismatch_values],
    rtol=1e-6,
    atol=1e-8,
)


Adjusted-close reference mismatches repaired: 10417


,ticker,ts,adj_close_original,adj_close_reference_replacement
13,AAPL,1983-11-15,1.483332,0.067968
124,AAPL,2000-06-13,11.165450,0.712530
130,AAPL,2001-04-06,2.424688,0.310500
178,AAPL,2006-12-12,21.941584,2.600000
252,AAPL,2017-01-05,284.046509,27.030000
272,AAPL,2019-09-12,797.720044,53.970000
311,AMGN,1984-02-03,1.806668,0.110760
404,AMGN,1997-11-10,197.261037,9.190000
413,AMGN,1998-10-15,114.571528,13.600000
483,AMGN,2008-09-05,347.282423,42.820000


## Step 5:  Validate Prices and OHLC Relationships

In [27]:
price_cols = ["open", "high", "low", "close", "adj_close"]

print("Invalid prices by column:")
invalid_price_counts = {}

for col in price_cols:
    values = pd.to_numeric(df_clean[col], errors="coerce")
    mask = (~np.isfinite(values)) | (values <= 0)
    invalid_price_counts[col] = int(mask.sum())

display(
    pd.Series(invalid_price_counts, name="invalid_count")
    .to_frame()
)

# Check raw OHLC relationships.
relationship_checks = {
    "high < close": df_clean["high"] < df_clean["close"],
    "low > close": df_clean["low"] > df_clean["close"],
    "low > open": df_clean["low"] > df_clean["open"],
    "high < open": df_clean["high"] < df_clean["open"],
    "low > high": df_clean["low"] > df_clean["high"],
}

print("\nImpossible OHLC relationships:")
for name, mask in relationship_checks.items():
    print(f"{name}: {int(mask.fillna(False).sum())}")

# Display rows that need investigation.
invalid_mask = pd.Series(False, index=df_clean.index)

for col in price_cols:
    values = pd.to_numeric(df_clean[col], errors="coerce")
    invalid_mask |= (~np.isfinite(values)) | (values <= 0)

for mask in relationship_checks.values():
    invalid_mask |= mask.fillna(False)

print("\nRows requiring price investigation:",
      int(invalid_mask.sum()))

display(
    df_clean.loc[invalid_mask]
    .sort_values(["ticker", "ts"])
    .head(25)
)

Invalid prices by column:


,invalid_count
open,5
high,0
low,0
close,0
adj_close,0



Impossible OHLC relationships:
high < close: 0
low > close: 0
low > open: 5
high < open: 0
low > high: 0

Rows requiring price investigation: 5


,ts,open,high,low,close,adj_close,volume,ticker,adj_close_repaired,adj_close_missing_repaired,adj_close_original,adj_close_reference_mismatch
124504,1962-01-02,0.0,3.30000,3.24000,3.30000,0.335750,105840,CVX,False,False,NaN,False
202784,1962-01-02,0.0,0.22338,0.22222,0.22338,0.064012,0,JNJ,False,False,NaN,False
247201,1966-07-05,0.0,0.27366,0.26749,0.26955,0.118090,388800,MCD,False,False,NaN,False
276726,1962-01-02,0.0,0.39703,0.39317,0.39537,0.068447,633830,MRK,False,False,NaN,False
318710,1962-01-02,0.0,1.44000,1.42000,1.43000,0.259300,192000,PG,False,False,NaN,False


### Price Validation — Findings

The cleaned working dataset was checked for non-finite or nonpositive prices and impossible raw OHLC relationships. Violations were identified for review before further repairs.

Adjusted closing prices were recovered separately using matched reference observations. Raw OHLC relationships were evaluated using the original raw price columns; adjusted closing prices are not required to fall within the raw daily high-low range.

The next step is to investigate any remaining violations and apply documented repairs only where justified.

## Step 6: Investigate Invalid Opening Prices

In [28]:
# Load raw OHLC values from the reference table.
with sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True) as conn:
    reference_ohlc = pd.read_sql_query(
        """
        SELECT ts, ticker, open, high, low, close, adj_close, volume
        FROM ohlc
        """,
        conn,
    )

reference_ohlc["ticker"] = (
    reference_ohlc["ticker"].map(normalize_ticker)
)
reference_ohlc["ts"] = pd.to_datetime(
    reference_ohlc["ts"], errors="coerce"
)

# Identify rows with invalid opening prices.
bad_open_mask = (
    ~np.isfinite(df_clean["open"])
    | (df_clean["open"] <= 0)
)

bad_open_rows = df_clean.loc[
    bad_open_mask,
    ["ticker", "ts", "open", "high", "low", "close", "volume"],
].copy()

# Match reference observations using ticker and date.
reference_open_lookup = reference_ohlc.set_index(
    ["ticker", "ts"]
)["open"]

bad_keys = pd.MultiIndex.from_frame(
    bad_open_rows[["ticker", "ts"]]
)

bad_open_rows["reference_open"] = (
    reference_open_lookup.reindex(bad_keys).to_numpy()
)

bad_open_rows["reference_open_valid"] = (
    np.isfinite(bad_open_rows["reference_open"])
    & (bad_open_rows["reference_open"] > 0)
)

display(bad_open_rows)

print(
    "Invalid openings with valid reference values:",
    int(bad_open_rows["reference_open_valid"].sum()),
)
print(
    "Invalid openings unresolved by reference:",
    int((~bad_open_rows["reference_open_valid"]).sum()),
)

,ticker,ts,open,high,low,close,volume,reference_open,reference_open_valid
124504,CVX,1962-01-02,0.0,3.30000,3.24000,3.30000,105840,0.0,False
202784,JNJ,1962-01-02,0.0,0.22338,0.22222,0.22338,0,0.0,False
247201,MCD,1966-07-05,0.0,0.27366,0.26749,0.26955,388800,0.0,False
276726,MRK,1962-01-02,0.0,0.39703,0.39317,0.39537,633830,0.0,False
318710,PG,1962-01-02,0.0,1.44000,1.42000,1.43000,192000,0.0,False


Invalid openings with valid reference values: 0
Invalid openings unresolved by reference: 5


### Invalid Opening Price Investigation — Findings

The five observations with zero opening prices were compared with the reference OHLC table using exact ticker-date matches.

Reference values are used to guide repair only when the corresponding opening price is valid and positive. Any unresolved observation will be tracked separately and excluded from the retained clean dataset if no defensible correction is available.

The source database remains unchanged.

## Step 7: Repair Invalid opening prices

In [29]:
# Track rows that require an opening-price repair.
bad_open_mask = (
    ~np.isfinite(df_clean["open"])
    | (df_clean["open"] <= 0)
)

df_clean["open_repaired"] = False

# Convert invalid openings to missing before applying the repair policy.
df_clean.loc[bad_open_mask, "open"] = np.nan

# Forward-fill only within each ticker, in chronological order.
df_clean = df_clean.sort_values(
    ["ticker", "ts"]
).copy()

previously_missing_open = df_clean["open"].isna()

df_clean["open"] = (
    df_clean.groupby("ticker", sort=False)["open"].ffill()
)

# Record which invalid openings were successfully repaired.
repaired_open_mask = previously_missing_open & df_clean["open"].notna()
df_clean.loc[repaired_open_mask, "open_repaired"] = True

# Rows still missing an opening price cannot be retained as valid OHLC data.
unresolved_open_mask = df_clean["open"].isna()

quarantined_open_rows = df_clean.loc[
    unresolved_open_mask
].copy()

# Report results.
print("Invalid openings before repair:", int(bad_open_mask.sum()))
print("Opening prices repaired by forward-fill:",
      int(df_clean["open_repaired"].sum()))
print("Rows quarantined due to unresolved opening:",
      len(quarantined_open_rows))

display(
    quarantined_open_rows[
        ["ticker", "ts", "open", "high", "low", "close", "volume"]
    ]
)

assert (
    df_clean.loc[~unresolved_open_mask, "open"] > 0
).all()

Invalid openings before repair: 5
Opening prices repaired by forward-fill: 0
Rows quarantined due to unresolved opening: 5


,ticker,ts,open,high,low,close,volume
124504,CVX,1962-01-02,NaN,3.30000,3.24000,3.30000,105840
202784,JNJ,1962-01-02,NaN,0.22338,0.22222,0.22338,0
247201,MCD,1966-07-05,NaN,0.27366,0.26749,0.26955,388800
276726,MRK,1962-01-02,NaN,0.39703,0.39317,0.39537,633830
318710,PG,1962-01-02,NaN,1.44000,1.42000,1.43000,192000


### Opening-Price Repair — Findings

Invalid opening prices were converted to missing values and handled using within-ticker forward-filling. Successfully repaired values were marked with `open_repaired = True`.

Any observation that remained without a valid opening price was placed in a separate quarantine dataset. These observations will be excluded from the final retained clean dataset unless a defensible correction becomes available.

The repair process preserves the distinction between repaired observations and unresolved data-quality problems.

### Step 8: Quarantine Unresolved Opening Prices

In [30]:
# Preserve the unresolved observations separately.
quarantined_rows = quarantined_open_rows.copy()

# Exclude unresolved rows from the retained working dataset.
df_clean = df_clean.loc[~unresolved_open_mask].copy()

print("Rows in quarantine:", len(quarantined_rows))
print("Rows retained for cleaning:", len(df_clean))
print("Missing opening prices in retained data:",
      int(df_clean["open"].isna().sum()))

assert len(quarantined_rows) == 5
assert df_clean["open"].notna().all()
assert (df_clean["open"] > 0).all()

print("\nQuarantine validation passed.")

Rows in quarantine: 5
Rows retained for cleaning: 368246
Missing opening prices in retained data: 0

Quarantine validation passed.


### Quarantine — Findings

Five observations with unresolved opening prices were separated from the retained working dataset. The quarantined rows remain available for audit and have not been silently discarded.

The retained dataset now contains no missing or nonpositive opening prices. Further validation will check the remaining raw OHLC relationships and other data-quality requirements.

### Step 9: Validate OHLC Relationships and Volume

After quarantining unresolved opening prices, the retained dataset is checked again for valid raw OHLC relationships and volume values.

For each observation, the following conditions must hold:

- `low <= open <= high`
- `low <= close <= high`
- All raw and adjusted prices must be finite and strictly positive.
- Volume must be finite and nonnegative.

Zero volume is not automatically treated as invalid because it may reflect the source data. Such observations are flagged for review rather than removed automatically.

In [31]:
# Validate raw OHLC relationships.
ohlc_checks = {
    "low > open": df_clean["low"] > df_clean["open"],
    "high < open": df_clean["high"] < df_clean["open"],
    "low > close": df_clean["low"] > df_clean["close"],
    "high < close": df_clean["high"] < df_clean["close"],
    "low > high": df_clean["low"] > df_clean["high"],
}

print("OHLC relationship violations:")
for name, mask in ohlc_checks.items():
    print(f"{name}: {int(mask.fillna(False).sum())}")

# Check all price columns.
price_cols = ["open", "high", "low", "close", "adj_close"]

print("\nInvalid prices by column:")
for col in price_cols:
    values = pd.to_numeric(df_clean[col], errors="coerce")
    invalid = (~np.isfinite(values)) | (values <= 0)
    print(f"{col}: {int(invalid.sum())}")

# Check volume.
volume = pd.to_numeric(df_clean["volume"], errors="coerce")
invalid_volume = (~np.isfinite(volume)) | (volume < 0)
zero_volume = volume == 0

print("\nInvalid volume rows:", int(invalid_volume.sum()))
print("Zero-volume rows:", int(zero_volume.sum()))

assert all(
    int(mask.fillna(False).sum()) == 0
    for mask in ohlc_checks.values()
)
assert all(
    np.isfinite(df_clean[col]).all()
    and (df_clean[col] > 0).all()
    for col in price_cols
)
assert not invalid_volume.any()

print("\nOHLC and volume validation passed.")

OHLC relationship violations:
low > open: 0
high < open: 0
low > close: 0
high < close: 0
low > high: 0

Invalid prices by column:
open: 0
high: 0
low: 0
close: 0
adj_close: 0

Invalid volume rows: 0
Zero-volume rows: 74

OHLC and volume validation passed.


### OHLC and Volume Validation — Findings

The retained dataset was rechecked after quarantine. The validation assessed all required OHLC relationships, finite positive prices, and finite nonnegative volume.

Zero-volume observations were counted separately and retained for review. They were not automatically interpreted as missing trading sessions or invalid prices.

The results establish whether the retained dataset satisfies the basic price and volume constraints before final cleaning and reporting.

### Step 10: Review Extreme Returns

Simple close-to-close returns are calculated separately for each ticker after sorting observations chronologically. Returns are flagged when they differ from the ticker's mean return by more than three sample standard deviations.

These flags are diagnostic rather than proof of incorrect data. Extreme movements will not be automatically repaired because legitimate corporate actions and market events can cause large returns.

The flagged observations will be retained for review unless independent evidence supports a correction or the observations must be quarantined for another unresolved data-quality issue.

In [32]:
# Sort by ticker and date before calculating returns.
df_clean = df_clean.sort_values(
    ["ticker", "ts"]
).copy()

df_clean["simple_return"] = (
    df_clean.groupby("ticker")["close"]
    .pct_change(fill_method=None)
)

extreme_return_records = []

for ticker, group in df_clean.groupby("ticker", sort=False):
    returns = group["simple_return"]
    valid_returns = returns[np.isfinite(returns)]

    if len(valid_returns) < 2:
        continue

    mean_return = valid_returns.mean()
    std_return = valid_returns.std(ddof=1)

    if not np.isfinite(std_return) or std_return == 0:
        continue

    mask = (
        returns.notna()
        & np.isfinite(returns)
        & ((returns - mean_return).abs() > 3 * std_return)
    )

    flagged = group.loc[mask]

    for idx, row in flagged.iterrows():
        extreme_return_records.append({
            "ticker": ticker,
            "ts": row["ts"],
            "close": row["close"],
            "simple_return": row["simple_return"],
            "mean_return": mean_return,
            "std_return": std_return,
        })

extreme_returns = pd.DataFrame(extreme_return_records)

print("Extreme-return observations:", len(extreme_returns))

if not extreme_returns.empty:
    display(
        extreme_returns
        .assign(
            return_pct=lambda x: x["simple_return"] * 100
        )
        .sort_values(
            "simple_return",
            key=lambda s: s.abs(),
            ascending=False,
        )
        .head(20)
    )
else:
    print("No extreme returns were detected.")

Extreme-return observations: 5057


,ticker,ts,close,simple_return,mean_return,std_return,return_pct
4430,TRV,1981-08-10,12.060000,1.000000,0.000573,0.024232,100.000000
4426,TRV,1976-10-28,9.220000,1.000000,0.000573,0.024232,100.000000
4428,TRV,1978-12-20,8.030000,0.997512,0.000573,0.024232,99.751244
3066,MCD,1968-01-02,0.174900,-0.753620,0.000675,0.018520,-75.362033
98,AAPL,2000-09-29,0.459820,-0.518695,0.001074,0.027846,-51.869452
4431,TRV,1981-08-11,6.030000,-0.500000,0.000573,0.024232,-50.000000
4427,TRV,1976-10-29,4.690000,-0.491323,0.000573,0.024232,-49.132321
4429,TRV,1978-12-21,4.090000,-0.490660,0.000573,0.024232,-49.066002
3955,NVDA,2000-03-07,0.173570,0.424106,0.001966,0.037879,42.410568
3997,NVDA,2004-08-06,0.078584,-0.352314,0.001966,0.037879,-35.231435


### Extreme-Return Review — Findings

Extreme returns were recalculated after duplicate removal and quarantine of observations with unresolved opening prices. The analysis uses the retained dataset and a ticker-specific three-standard-deviation rule.

Flagged observations are retained for further investigation. No return was automatically corrected or removed solely because it exceeded the statistical threshold. Any subsequent repair would require supporting evidence and documented justification.

### Extreme-Return Investigation — Findings

The three-standard-deviation screen identified unusually large close-to-close returns across several tickers. The largest movements include approximately +100% and -50% returns for TRV, as well as large movements for MCD, AAPL, NVDA, and other tickers.

These observations require investigation because corporate actions, historical price adjustments, or data errors may produce large returns. Statistical flags alone do not establish the cause of a movement.

No extreme-return observations were automatically repaired or removed. The flagged observations are retained for audit, and any proposed correction must be supported by independent evidence or a documented, defensible repair rule.

In [33]:
print("Total extreme-return observations:", len(extreme_returns))

if not extreme_returns.empty:
    print("\nFlagged observations by ticker:")
    display(
        extreme_returns.groupby("ticker")
        .size()
        .rename("extreme_return_count")
        .sort_values(ascending=False)
        .reset_index()
    )

    print("\nLargest absolute returns:")
    display(
        extreme_returns.assign(
            return_pct=extreme_returns["simple_return"] * 100
        )
        .assign(
            abs_return=lambda x: x["simple_return"].abs()
        )
        .sort_values("abs_return", ascending=False)
        .head(10)[
            ["ticker", "ts", "simple_return", "return_pct",
             "mean_return", "std_return"]
        ]
    )

Total extreme-return observations: 5057

Flagged observations by ticker:


,ticker,extreme_return_count
0,IBM,228
1,KO,226
2,HON,222
3,CAT,221
4,MMM,217
5,DIS,214
6,BA,210
7,WMT,209
8,JPM,207
9,JNJ,201



Largest absolute returns:


,ticker,ts,simple_return,return_pct,mean_return,std_return
4430,TRV,1981-08-10,1.000000,100.000000,0.000573,0.024232
4426,TRV,1976-10-28,1.000000,100.000000,0.000573,0.024232
4428,TRV,1978-12-20,0.997512,99.751244,0.000573,0.024232
3066,MCD,1968-01-02,-0.753620,-75.362033,0.000675,0.018520
98,AAPL,2000-09-29,-0.518695,-51.869452,0.001074,0.027846
4431,TRV,1981-08-11,-0.500000,-50.000000,0.000573,0.024232
4427,TRV,1976-10-29,-0.491323,-49.132321,0.000573,0.024232
4429,TRV,1978-12-21,-0.490660,-49.066002,0.000573,0.024232
3955,NVDA,2000-03-07,0.424106,42.410568,0.001966,0.037879
3997,NVDA,2004-08-06,-0.352314,-35.231435,0.001966,0.037879


### Step 11: Investigate Zero-Volume Observations

Zero-volume observations are flagged for investigation but are not automatically removed or treated as missing trading days.

The volume values are compared with the reference table using exact ticker-date matches. Matching reference values will help determine whether zero volume is also present in the reference data or whether the homework dataset differs.

Zero-volume observations will be retained unless there is sufficient evidence to justify a different treatment.

In [34]:
# Identify zero-volume observations in the retained dataset.
zero_volume_mask = df_clean["volume"] == 0

zero_volume_rows = df_clean.loc[
    zero_volume_mask,
    ["ticker", "ts", "volume"],
].copy()

# Match volume values from the reference table.
reference_volume_lookup = reference_ohlc.set_index(
    ["ticker", "ts"]
)["volume"]

zero_keys = pd.MultiIndex.from_frame(
    zero_volume_rows[["ticker", "ts"]]
)

zero_volume_rows["reference_volume"] = (
    reference_volume_lookup.reindex(zero_keys).to_numpy()
)

zero_volume_rows["reference_match"] = (
    zero_volume_rows["reference_volume"].notna()
)

zero_volume_rows["reference_also_zero"] = (
    zero_volume_rows["reference_volume"] == 0
)

print("Zero-volume observations:", len(zero_volume_rows))
print(
    "With matching reference observations:",
    int(zero_volume_rows["reference_match"].sum()),
)
print(
    "Also zero in reference:",
    int(zero_volume_rows["reference_also_zero"].sum()),
)
print(
    "Positive volume in reference:",
    int(
        (
            zero_volume_rows["reference_volume"].notna()
            & (zero_volume_rows["reference_volume"] > 0)
        ).sum()
    ),
)
print(
    "Without matching reference volume:",
    int((~zero_volume_rows["reference_match"]).sum()),
)

display(zero_volume_rows.head(25))

Zero-volume observations: 74
With matching reference observations: 74
Also zero in reference: 74
Positive volume in reference: 0
Without matching reference volume: 0


,ticker,ts,volume,reference_volume,reference_match,reference_also_zero
67725,AXP,1972-12-14,0,0,True,True
177087,HON,1982-09-23,0,0,True,True
177088,HON,1982-09-24,0,0,True,True
190211,IBM,1973-10-09,0,0,True,True
34149,IBM,1975-01-27,0,0,True,True
192253,IBM,1982-01-08,0,0,True,True
202857,JNJ,1962-04-17,0,0,True,True
202979,JNJ,1962-09-28,0,0,True,True
202991,JNJ,1962-10-15,0,0,True,True
203650,JNJ,1965-07-01,0,0,True,True


### Zero-Volume Investigation — Findings

Zero-volume observations were compared with reference volume values for matching ticker-date pairs. The comparison distinguishes observations where the reference also reports zero volume from cases where the reference reports positive volume or has no usable matching value.

Zero-volume observations are not automatically removed or replaced. Their treatment depends on the reference comparison and the broader data-quality evidence. The original source table remains unchanged.

The dataset contains 74 zero-volume observations. All 74 have matching ticker-date observations in the reference table, and the reference also reports zero volume for every one of them.

These observations were retained because the reference data supports the recorded values. No volume values were changed, and no observations were removed on the basis of zero volume alone.

### Step 12: Cleaning Audit and Before-and-After Statistics

In [35]:
# Preserve the original versions of the quarantined rows.
quarantine_original = df_before_cleaning.loc[
    quarantined_rows.index
].copy()

quarantine_original["quarantine_reason"] = (
    "Invalid opening price; no valid reference value "
    "or prior valid opening price available"
)

# Confirm the original invalid values remain available for audit.
assert len(quarantine_original) == 5
assert (quarantine_original["open"] == 0).all()

# Record the before-and-after counts.
cleaning_audit = pd.DataFrame([
    {
        "metric": "Rows before cleaning",
        "count": len(df_before_cleaning),
    },
    {
        "metric": "Exact duplicate rows removed",
        "count": duplicate_rows_removed,
    },
    {
        "metric": "Missing adjusted-close values recovered",
        "count": int(df_clean["adj_close_missing_repaired"].sum()),
    },
    {
        "metric": "Adjusted-close reference mismatches repaired",
        "count": int(df_clean["adj_close_reference_mismatch"].sum()),
    },
    {
        "metric": "Opening prices repaired by forward-fill",
        "count": int(df_clean["open_repaired"].sum()),
    },
    {
        "metric": "Rows quarantined",
        "count": len(quarantined_rows),
    },
    {
        "metric": "Rows retained",
        "count": len(df_clean),
    },
    {
        "metric": "Zero-volume rows retained",
        "count": int((df_clean["volume"] == 0).sum()),
    },
    {
        "metric": "Extreme-return flags",
        "count": len(extreme_returns),
    },
])

display(cleaning_audit)

print("Quarantine records preserved:", len(quarantine_original))
print("Repair flags:")
print("  adj_close_repaired (all adjusted-close repairs):",
      int(df_clean["adj_close_repaired"].sum()))
print("  adj_close_missing_repaired:",
      int(df_clean["adj_close_missing_repaired"].sum()))
print("  adj_close_reference_mismatch:",
      int(df_clean["adj_close_reference_mismatch"].sum()))
print("  open_repaired:",
      int(df_clean["open_repaired"].sum()))

assert len(df_clean) == 368246
assert len(quarantined_rows) == 5
assert int(df_clean["adj_close_missing_repaired"].sum()) == 18374
assert int(df_clean["adj_close_repaired"].sum()) == (
    18374 + int(df_clean["adj_close_reference_mismatch"].sum())
)
assert int(df_clean["open_repaired"].sum()) == 0

print("\nCleaning audit checks passed.")

,metric,count
0,Rows before cleaning,397711
1,Exact duplicate rows removed,29460
2,Missing adjusted-close values recovered,18374
3,Adjusted-close reference mismatches repaired,10417
4,Opening prices repaired by forward-fill,0
5,Rows quarantined,5
6,Rows retained,368246
7,Zero-volume rows retained,74
8,Extreme-return flags,5057


Quarantine records preserved: 5
Repair flags:
  adj_close_repaired (all adjusted-close repairs): 28791
  adj_close_missing_repaired: 18374
  adj_close_reference_mismatch: 10417
  open_repaired: 0

Cleaning audit checks passed.


### Cleaning Audit — Findings

The cleaning process began with 397,711 rows. Removing 29,460 exact duplicate observations left 368,251 rows. Five observations with unresolved opening prices were quarantined, leaving 368,246 rows in the retained dataset.

All 18,374 missing adjusted closing prices were recovered using matching ticker-date observations from the reference table. Remaining valid-but-inconsistent adjusted-close values were also compared against the exact ticker-date reference and repaired when a valid reference value was available; the original values and mismatch flags are retained for audit. No opening prices were repaired through forward-filling. The 74 zero-volume observations were retained because all matched the reference data.

The extreme-return screen identified 5,057 observations for further investigation. These were not automatically modified or removed because statistical outliers are not necessarily data errors.

Repair flags and quarantine records were preserved to support reproducibility and auditing. The original source data remains unchanged.

In [36]:
# Step 30: Final Data Validation

required_columns = [
    "ticker", "ts", "open", "high", "low",
    "close", "adj_close", "volume"
]

# 1. Required columns exist
assert all(col in df_clean.columns for col in required_columns)

# 2. No missing values in required columns
assert not df_clean[required_columns].isna().any().any()

# 3. Unique ticker-date combinations
assert not df_clean.duplicated(
    subset=["ticker", "ts"]
).any()

# 4. All prices are finite and strictly positive
price_cols = ["open", "high", "low", "close", "adj_close"]

for col in price_cols:
    values = df_clean[col].to_numpy(dtype=float)

    assert np.isfinite(values).all(), (
        f"Non-finite values found in {col}"
    )
    assert (values > 0).all(), (
        f"Nonpositive values found in {col}"
    )

# 5. Valid OHLC relationships
assert (df_clean["low"] <= df_clean["open"]).all()
assert (df_clean["open"] <= df_clean["high"]).all()
assert (df_clean["low"] <= df_clean["close"]).all()
assert (df_clean["close"] <= df_clean["high"]).all()

# 6. Volume is finite and nonnegative
volume = df_clean["volume"].to_numpy(dtype=float)

assert np.isfinite(volume).all()
assert (volume >= 0).all()

# 7. Repair flags and audit columns are preserved
assert "adj_close_repaired" in df_clean.columns
assert "adj_close_missing_repaired" in df_clean.columns
assert "adj_close_reference_mismatch" in df_clean.columns
assert "adj_close_original" in df_clean.columns
assert "open_repaired" in df_clean.columns

# 8. Quarantined observations remain separate
assert len(quarantined_rows) == 5
assert len(df_clean) + len(quarantined_rows) == (
    len(df_before_cleaning) - duplicate_rows_removed
)

# 9. Expected row counts
assert len(df_clean) == 368246
assert len(quarantined_rows) == 5

print("All final validation checks passed!")
print(f"Retained rows: {len(df_clean):,}")
print(f"Quarantined rows: {len(quarantined_rows):,}")
print(f"Unique tickers: {df_clean['ticker'].nunique()}")
print(
    f"Date range: {df_clean['ts'].min().date()} "
    f"to {df_clean['ts'].max().date()}"
)
print(
    "Missing adjusted close:",
    int(df_clean["adj_close"].isna().sum())
)
print(
    "Duplicate ticker-date keys:",
    int(df_clean.duplicated(["ticker", "ts"]).sum())
)

All final validation checks passed!
Retained rows: 368,246
Quarantined rows: 5
Unique tickers: 30
Date range: 1962-01-02 to 2024-12-06
Missing adjusted close: 0
Duplicate ticker-date keys: 0


### Final Validation — Findings

The retained dataset passed the final validation checks for required fields, missing values, ticker-date uniqueness, finite positive prices, valid OHLC relationships, and finite nonnegative volume.

A total of 368,246 observations were retained, while five observations with unresolved opening prices remained in quarantine. Repair flags were preserved, and the original source table was not modified.

The validated dataset is ready for the final missing-trading-day reconciliation and export to a separate SQLite table.

### Step 14: Missing-Trading-Day Reconciliation

In [37]:
# Reconstruct the dataset after exact duplicate removal,
# before quarantining invalid opening prices.
df_deduplicated = df_before_cleaning.drop_duplicates(
    keep="first"
).copy()

df_deduplicated["ticker"] = (
    df_deduplicated["ticker"].map(normalize_ticker)
)
df_deduplicated["ts"] = pd.to_datetime(
    df_deduplicated["ts"], errors="coerce"
)

# Compare distinct ticker-date keys.
actual_keys = set(
    zip(df_deduplicated["ticker"], df_deduplicated["ts"])
)

reference_keys = set(
    zip(reference_df["ticker"], reference_df["ts"])
)

missing_keys = sorted(
    reference_keys - actual_keys,
    key=lambda item: (item[0], item[1])
)

missing_trading_days_final = pd.DataFrame(
    [
        {"ticker": ticker, "issue_date": ts.strftime("%Y-%m-%d")}
        for ticker, ts in missing_keys
    ],
    columns=["ticker", "issue_date"],
)

print("Distinct ticker-date keys before quarantine:", len(actual_keys))
print("Distinct reference ticker-date keys:", len(reference_keys))
print("Missing reference ticker-date keys:", len(missing_trading_days_final))

if not missing_trading_days_final.empty:
    display(
        missing_trading_days_final
        .groupby("ticker")
        .size()
        .rename("missing_trading_days")
        .reset_index()
        .sort_values("missing_trading_days", ascending=False)
    )
    display(missing_trading_days_final.head(20))
else:
    print("No missing reference trading dates detected.")

# Verify the five quarantined observations are excluded for a different reason.
print("\nQuarantined observations:", len(quarantined_rows))
print("Quarantine reason: unresolved invalid opening prices")

Distinct ticker-date keys before quarantine: 368251
Distinct reference ticker-date keys: 368251
Missing reference ticker-date keys: 0
No missing reference trading dates detected.

Quarantined observations: 5
Quarantine reason: unresolved invalid opening prices


### Missing-Trading-Day Reconciliation — Findings

The deduplicated dataset contains 368,251 distinct ticker-date combinations, matching all 368,251 distinct ticker-date combinations in the reference table. No missing reference trading dates were detected.

The five quarantined observations were excluded because their opening prices were invalid and could not be reliably repaired. They were not missing dates in the original dataset.

This confirms that the original deduplicated dataset has complete coverage of the supplied reference ticker-date keys.

## Save the cleaned dataset to SQLite

In [38]:
# Keep the clean table's expected fields and repair flags.
output_columns = [
    "ts", "open", "high", "low", "close",
    "adj_close", "volume", "ticker",
    "adj_close_repaired", "adj_close_missing_repaired",
    "adj_close_reference_mismatch", "adj_close_original",
    "open_repaired",
]

clean_output = df_clean[output_columns].copy()
quarantine_output = quarantine_original.copy()

# Store dates as ISO YYYY-MM-DD strings.
clean_output["ts"] = pd.to_datetime(
    clean_output["ts"]
).dt.strftime("%Y-%m-%d")

if "ts" in quarantine_output.columns:
    quarantine_output["ts"] = pd.to_datetime(
        quarantine_output["ts"]
    ).dt.strftime("%Y-%m-%d")

# Open the database in writable mode, but protect source tables.
with sqlite3.connect(DB_PATH) as conn:
    existing_tables = {
        row[0]
        for row in conn.execute(
            "SELECT name FROM sqlite_master WHERE type='table'"
        ).fetchall()
    }

    protected_tables = {"ohlc_hw", "ohlc", "market_caps"}
    assert protected_tables.issubset(existing_tables)

    # Save only the output tables.
    clean_output.to_sql(
        "ohlc_hw_cleaned",
        conn,
        if_exists="replace",
        index=False,
    )

    quarantine_output.to_sql(
        "ohlc_hw_quarantine",
        conn,
        if_exists="replace",
        index=False,
    )

    # Verify stored row counts.
    saved_clean_count = conn.execute(
        "SELECT COUNT(*) FROM ohlc_hw_cleaned"
    ).fetchone()[0]

    saved_quarantine_count = conn.execute(
        "SELECT COUNT(*) FROM ohlc_hw_quarantine"
    ).fetchone()[0]

    assert saved_clean_count == len(clean_output)
    assert saved_quarantine_count == len(quarantine_output)

    # Confirm source tables still exist.
    remaining_tables = {
        row[0]
        for row in conn.execute(
            "SELECT name FROM sqlite_master WHERE type='table'"
        ).fetchall()
    }

    assert protected_tables.issubset(remaining_tables)

print("Cleaned table saved:", saved_clean_count)
print("Quarantine table saved:", saved_quarantine_count)
print("Original source tables preserved.")

Cleaned table saved: 368246
Quarantine table saved: 5
Original source tables preserved.


### SQLite Export — Findings

The validated retained dataset was saved to `ohlc_hw_cleaned`, and unresolved observations were saved separately to `ohlc_hw_quarantine`.

Dates are stored in ISO `YYYY-MM-DD` format. The cleaned table retains separate flags for missing adjusted-close recovery, reference-mismatch repairs, and opening-price repairs. Original adjusted-close values for mismatch repairs are retained in `adj_close_original` for auditability.

The original `ohlc_hw`, `ohlc`, and `market_caps` tables were preserved. The stored row counts were checked against the in-memory datasets to confirm that the export completed successfully.

## Part 1 Conclusion — Data Cleaning

The initial `ohlc_hw` dataset contained 397,711 rows across 30 tickers. The cleaning process removed 29,460 exact duplicate observations, recovered 18,374 missing adjusted closing prices using exact ticker-date matches in the reference table, checked adjusted-close values against the reference for valid-but-inconsistent values, and quarantined five observations with unresolved zero opening prices.

The retained dataset contains 368,246 rows. Final validation confirmed unique ticker-date keys, complete required fields, finite positive prices, valid OHLC relationships, and finite nonnegative volume. All 74 zero-volume observations were retained because they matched the reference table. The 5,057 extreme-return flags were retained for investigation rather than automatically treated as errors.

A comparison against the reference table found no missing ticker-date combinations in the deduplicated dataset. The retained dataset and quarantine records were saved to separate SQLite tables, `ohlc_hw_cleaned` and `ohlc_hw_quarantine`, without overwriting the original source tables.

The cleaning process preserves repair flags, original adjusted-close values for reference-mismatch repairs, and quarantine records to support reproducibility, auditability, and downstream analysis.